In [57]:
import requests
import os
from dotenv import load_dotenv
import json
load_dotenv()



True

In [2]:
jira_base_url = os.environ.get("JIRA_URL")
jira_api_key = os.environ.get("API_KEY")
jira_email = os.environ.get("EMAIL")

In [61]:
def get_all_epics(project_key):
    """
    Get all epics for a given project key.
    
    Args:
        project_key (str): The Jira project key (e.g., 'MH')
    
    Returns:
        list: List of dictionaries containing epic information:
            - epic_summary: Epic summary/title
            - epic_key: Epic key (e.g., 'MH-1')
            - epic_description: Epic description (plain text)
            - epic_created_at: Epic creation date
            - epic_start_date: Epic start date (if exists)
            - epic_due_date: Epic due date (if exists)
            - epic_status: Epic status (if exists)
    """
    import json
    
    # Step 1: Search for all epics in the project
    url = f"{jira_base_url}/rest/api/3/search/jql"
    headers = {
        "Accept": "application/json"
    }
    auth = (jira_email, jira_api_key)
    query = {
        'jql': f'project = {project_key} AND issuetype = Epic'
    }
    response = requests.request(
        "GET",
        url,
        headers=headers,
        params=query,
        auth=auth
    )
    response.raise_for_status()
    epic_data = json.loads(response.text)
    epic_ids = [issue['id'] for issue in epic_data.get('issues', [])]
    
    # Step 2: Get full details for each epic
    all_epics = []
    for epic_id in epic_ids:
        url = f"{jira_base_url}/rest/api/3/issue/{epic_id}"
        headers = {
            "Accept": "application/json"
        }
        auth = (jira_email, jira_api_key)
        epic_response = requests.request(
            "GET",
            url,
            headers=headers,
            auth=auth
        )
        epic_response.raise_for_status()
        epic_details = json.loads(epic_response.text)
        
        # Extract fields
        fields = epic_details.get('fields', {})
        
        # Extract description (handle ADF format)
        description = ""
        description_field = fields.get('description')
        if description_field:
            if isinstance(description_field, dict):
                # ADF format - extract text recursively
                def extract_adf_text(adf_content):
                    if isinstance(adf_content, str):
                        return adf_content
                    if isinstance(adf_content, dict):
                        if adf_content.get("type") == "text":
                            return adf_content.get("text", "")
                        if "content" in adf_content:
                            return extract_adf_text(adf_content["content"])
                    if isinstance(adf_content, list):
                        return " ".join(extract_adf_text(item) for item in adf_content)
                    return ""
                description = extract_adf_text(description_field)
            else:
                description = str(description_field)
        
        # Extract status
        status = None
        status_field = fields.get('status')
        if status_field:
            status = status_field.get('name')
        
        # Build epic info dictionary
        epic_info = {
            'epic_summary': fields.get('summary', ''),
            'epic_key': epic_details.get('key', ''),
            'epic_description': description,
            'epic_created_at': fields.get('created', ''),
            'epic_start_date': fields.get('customfield_10014', None),  # Common epic start date field
            'epic_due_date': fields.get('duedate', None),
            'epic_status': status
        }
        
        all_epics.append(epic_info)
    
    return all_epics


In [ ]:
# CORRECTED create_jira_issue function with date handling
# Copy this code and replace your existing function in cell 11

def create_jira_issue(
    project_key,
    epic_key,
    summary,
    description,
    issue_type="Task",
    parent_key=None,
    start_date=None,
    due_date=None,
    assignee=None,
    story_points=None
):
    """
    Creates a new Jira issue or sub-task and links it to an Epic (if epic_key is provided).

    :param project_key: The key of the project (e.g., "MH").
    :param epic_key: The issue key of the parent epic (e.g., "MH-7").
    :param summary: The issue summary/title.
    :param description: The issue description.
    :param issue_type: The type of issue (e.g., "Task", "Story", "Bug").
    :param parent_key: Used for creating sub-tasks (requires a different payload structure).
    :param start_date: Start date in format 'YYYY-MM-DD' (optional).
    :param due_date: Due date in format 'YYYY-MM-DD' (optional).
    :param assignee: Assignee account ID (optional).
    :param story_points: Story points value (optional).
    :returns: The key of the newly created issue (e.g., "MH-8").
    """
    EPIC_LINK_FIELD = "customfield_10014"
    # Base URL for the Jira Cloud REST API v3
    url = f"{jira_base_url}/rest/api/3/issue"
    auth = (jira_email, jira_api_key)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json"
    }

    # Build the core payload
    fields = {
        "project": {
            "key": project_key
        },
        "summary": summary,
        "description": { # Jira Cloud uses Atlassian Document Format (ADF) for descriptions
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [
                        {
                            "type": "text",
                            "text": description
                        }
                    ]
                }
            ]
        },
        "issuetype": {
            "name": issue_type
        },
    }

    # Set Organizations field (customfield_10002) to empty array as it's required
    fields["customfield_10002"] = []

    # Add optional fields if provided
    if epic_key and not parent_key:
        # This is the crucial part: linking to the epic using the custom field ID
        fields[EPIC_LINK_FIELD] = epic_key

    if assignee:
        fields["assignee"] = {"accountId": assignee} # Use account ID in Jira Cloud

    # Add due date (standard Jira field)
    if due_date:
        fields["duedate"] = due_date  # Format: 'YYYY-MM-DD'
    
    # Add start date (may be a custom field in your Jira instance)
    if start_date:
        # Common custom field IDs for start date: customfield_10015, customfield_10013, etc.
        # Note: customfield_10014 is used for epic link, so we use a different field for start date
        # Adjust the field ID if needed for your Jira instance - you may need to check your Jira config
        fields["customfield_10015"] = start_date  # Format: 'YYYY-MM-DD'
        # If this doesn't work, check your Jira instance's custom field configuration for the start date field

    if story_points is not None:
        # Story Points is another custom field, ID will vary!
        fields["customfield_10016"] = story_points # Replace with your SP field ID if different
        
    if parent_key:
        # Note: Sub-tasks require a 'parent' field and a sub-task issuetype name
        fields["parent"] = {"key": parent_key}

    payload = json.dumps({"fields": fields})

    try:
        response = requests.post(
            url,
            data=payload,
            headers=headers,
            auth=auth
        )
        
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        response_data = response.json()
        print(f"Successfully created issue: {response_data['key']}")
        return response_data['key'] # Returns the issue key, e.g., "MH-8"

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        if 'response' in locals() and response is not None:
             print(f"Response Status Code: {response.status_code}")
             print(f"Response Body: {response.text}")
        return None


In [ ]:
def update_jira_issue(
    issue_key,
    summary=None,
    description=None,
    assignee=None,
    priority=None,
    due_date=None,
    start_date=None,
    story_points=None,
    labels=None,
    status=None
):
    """
    Updates an existing Jira issue with the provided fields.
    
    Args:
        issue_key (str): The Jira issue key (e.g., 'MH-8')
        summary (str, optional): New issue summary/title
        description (str, optional): New issue description (plain text)
        assignee (str, optional): New assignee account ID
        priority (str, optional): New priority name (e.g., 'High', 'Medium', 'Low')
        due_date (str, optional): New due date in format 'YYYY-MM-DD'
        start_date (str, optional): New start date in format 'YYYY-MM-DD'
        story_points (int, optional): New story points value
        labels (list, optional): New list of labels (replaces existing labels)
        status (str, optional): New status name (e.g., 'In Progress', 'Done')
                                Note: Status changes use transitions, not direct field updates
    
    Returns:
        dict: Dictionary containing:
            - success: True if successful
            - updated_fields: List of fields that were updated
            - issue_key: The updated issue key
    """
    import json
    
    url = f"{jira_base_url}/rest/api/3/issue/{issue_key}"
    auth = (jira_email, jira_api_key)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json"
    }
    
    # Build the fields to update
    fields = {}
    updated_fields = []
    
    if summary is not None:
        fields["summary"] = summary
        updated_fields.append("summary")
    
    if description is not None:
        # Convert to ADF format
        fields["description"] = {
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [
                        {
                            "type": "text",
                            "text": description
                        }
                    ]
                }
            ]
        }
        updated_fields.append("description")
    
    if assignee is not None:
        # If assignee is empty string or None, unassign
        if assignee == "" or assignee is None:
            fields["assignee"] = None
        else:
            fields["assignee"] = {"accountId": assignee}
        updated_fields.append("assignee")
    
    if priority is not None:
        fields["priority"] = {"name": priority}
        updated_fields.append("priority")
    
    if due_date is not None:
        fields["duedate"] = due_date
        updated_fields.append("due_date")
    
    if start_date is not None:
        fields["customfield_10015"] = start_date  # Common start date field
        updated_fields.append("start_date")
    
    if story_points is not None:
        fields["customfield_10016"] = story_points  # Common story points field
        updated_fields.append("story_points")
    
    if labels is not None:
        fields["labels"] = labels if isinstance(labels, list) else [labels]
        updated_fields.append("labels")
    
    # Update the issue fields
    if fields:
        payload = json.dumps({"fields": fields})
        
        try:
            response = requests.put(
                url,
                data=payload,
                headers=headers,
                auth=auth
            )
            response.raise_for_status()
            print(f"Successfully updated issue {issue_key}")
            print(f"Updated fields: {', '.join(updated_fields)}")
        except requests.exceptions.RequestException as e:
            print(f"An error occurred updating fields: {e}")
            if 'response' in locals() and response is not None:
                print(f"Response Status Code: {response.status_code}")
                print(f"Response Body: {response.text}")
            return {
                "success": False,
                "error": str(e),
                "issue_key": issue_key
            }
    
    # Handle status transition separately (if provided)
    if status is not None:
        try:
            # Get available transitions
            transitions_url = f"{jira_base_url}/rest/api/3/issue/{issue_key}/transitions"
            transitions_response = requests.get(
                transitions_url,
                headers=headers,
                auth=auth
            )
            transitions_response.raise_for_status()
            transitions_data = transitions_response.json()
            
            # Find the transition ID for the target status
            transition_id = None
            for transition in transitions_data.get('transitions', []):
                if transition.get('to', {}).get('name', '').lower() == status.lower():
                    transition_id = transition.get('id')
                    break
            
            if transition_id:
                # Execute the transition
                transition_payload = json.dumps({
                    "transition": {
                        "id": transition_id
                    }
                })
                
                transition_response = requests.post(
                    transitions_url,
                    data=transition_payload,
                    headers=headers,
                    auth=auth
                )
                transition_response.raise_for_status()
                print(f"Successfully transitioned issue to '{status}'")
                updated_fields.append("status")
            else:
                print(f"Warning: Could not find transition to status '{status}'")
                print(f"Available transitions: {[t.get('to', {}).get('name') for t in transitions_data.get('transitions', [])]}")
        
        except requests.exceptions.RequestException as e:
            print(f"An error occurred updating status: {e}")
            if 'response' in locals() and transition_response is not None:
                print(f"Response Status Code: {transition_response.status_code}")
                print(f"Response Body: {transition_response.text}")
    
    return {
        "success": True,
        "updated_fields": updated_fields,
        "issue_key": issue_key
    }


In [ ]:
def get_all_issues(project_key, epic_key=None):
    """
    Get all issues for a given project key, optionally filtered by epic.
    
    Args:
        project_key (str): The Jira project key (e.g., 'MH')
        epic_key (str, optional): The epic key to filter issues (e.g., 'MH-7'). 
                                  If None, returns all issues in the project.
    
    Returns:
        list: List of dictionaries containing issue information:
            - issue_key: Issue key (e.g., 'MH-8')
            - issue_summary: Issue summary/title
            - issue_description: Issue description (plain text)
            - issue_type: Issue type (Story, Task, Bug, etc.)
            - issue_status: Issue status
            - issue_created_at: Issue creation date
            - issue_updated_at: Issue last updated date
            - issue_due_date: Issue due date (if exists)
            - issue_assignee: Assignee name (if exists)
            - issue_reporter: Reporter name
            - issue_priority: Priority (if exists)
            - epic_key: Epic key if linked to an epic
    """
    import json
    
    # Build JQL query
    if epic_key:
        # Filter by project and epic
        jql_query = f'project = {project_key} AND "Epic Link" = {epic_key}'
    else:
        # Get all issues in project (excluding epics)
        jql_query = f'project = {project_key} AND issuetype != Epic'
    
    # Step 1: Search for issues
    url = f"{jira_base_url}/rest/api/3/search/jql"
    headers = {
        "Accept": "application/json"
    }
    auth = (jira_email, jira_api_key)
    query = {
        'jql': jql_query
    }
    
    all_issues = []
    start_at = 0
    max_results = 100
    
    # Handle pagination
    while True:
        query['startAt'] = start_at
        query['maxResults'] = max_results
        
        response = requests.request(
            "GET",
            url,
            headers=headers,
            params=query,
            auth=auth
        )
        response.raise_for_status()
        search_data = json.loads(response.text)
        
        issue_ids = search_data.get('issues', [])
        if not issue_ids:
            return []


        return issue_ids


In [ ]:
def get_issue_details(issue_key):
    """
    Get detailed information for a specific Jira issue.
    
    Args:
        issue_key (str): The Jira issue key or ID (e.g., 'MH-8' or '10038')
    
    Returns:
        dict: Dictionary containing detailed issue information:
            - issue_key: Issue key (e.g., 'MH-8')
            - issue_id: Issue ID
            - issue_summary: Issue summary/title
            - issue_description: Issue description (plain text)
            - issue_type: Issue type (Story, Task, Bug, etc.)
            - issue_status: Issue status
            - issue_created_at: Issue creation date
            - issue_updated_at: Issue last updated date
            - issue_due_date: Issue due date (if exists)
            - issue_start_date: Issue start date (if exists)
            - issue_assignee: Assignee name (if exists)
            - issue_assignee_id: Assignee account ID (if exists)
            - issue_reporter: Reporter name
            - issue_priority: Priority (if exists)
            - epic_key: Epic key if linked to an epic
            - parent_key: Parent issue key (for sub-tasks)
            - story_points: Story points (if exists)
            - labels: List of labels
            - components: List of component names
    """
    import json
    
    # Get issue details
    url = f"{jira_base_url}/rest/api/3/issue/{issue_key}"
    headers = {
        "Accept": "application/json"
    }
    auth = (jira_email, jira_api_key)
    
    response = requests.request(
        "GET",
        url,
        headers=headers,
        auth=auth
    )
    response.raise_for_status()
    issue_data = json.loads(response.text)
    
    # Extract fields
    fields = issue_data.get('fields', {})
    
    # Extract description (handle ADF format)
    description = ""
    description_field = fields.get('description')
    if description_field:
        if isinstance(description_field, dict):
            # ADF format - extract text recursively
            def extract_adf_text(adf_content):
                if isinstance(adf_content, str):
                    return adf_content
                if isinstance(adf_content, dict):
                    if adf_content.get("type") == "text":
                        return adf_content.get("text", "")
                    if "content" in adf_content:
                        return extract_adf_text(adf_content["content"])
                if isinstance(adf_content, list):
                    return " ".join(extract_adf_text(item) for item in adf_content)
                return ""
            description = extract_adf_text(description_field)
        else:
            description = str(description_field)
    
    # Extract epic key if linked
    epic_key_linked = None
    epic_link_field = fields.get('customfield_10014')  # Common epic link field
    if epic_link_field:
        epic_key_linked = epic_link_field
    
    # Extract parent key (for sub-tasks)
    parent_key = None
    parent_field = fields.get('parent')
    if parent_field:
        parent_key = parent_field.get('key')
    
    # Extract start date (may be custom field)
    start_date = None
    start_date_field = fields.get('customfield_10015')  # Common start date field
    if start_date_field:
        start_date = start_date_field
    
    # Extract story points (may be custom field)
    story_points = None
    story_points_field = fields.get('customfield_10016')  # Common story points field
    if story_points_field:
        story_points = story_points_field
    
    # Extract labels
    labels = fields.get('labels', [])
    
    # Extract components
    components = [comp.get('name') for comp in fields.get('components', [])]
    
    # Extract assignee info
    assignee_name = None
    assignee_id = None
    assignee = fields.get('assignee')
    if assignee:
        assignee_name = assignee.get('displayName')
        assignee_id = assignee.get('accountId')
    
    # Build issue info dictionary
    issue_info = {
        'issue_key': issue_data.get('key'),
        'issue_id': issue_data.get('id'),
        'issue_summary': fields.get('summary', ''),
        'issue_description': description,
        'issue_type': fields.get('issuetype', {}).get('name', 'Unknown'),
        'issue_status': fields.get('status', {}).get('name', 'Unknown'),
        'issue_created_at': fields.get('created', ''),
        'issue_updated_at': fields.get('updated', ''),
        'issue_due_date': fields.get('duedate', None),
        'issue_start_date': start_date,
        'issue_assignee': assignee_name,
        'issue_assignee_id': assignee_id,
        'issue_reporter': fields.get('reporter', {}).get('displayName', 'Unknown') if fields.get('reporter') else 'Unknown',
        'issue_priority': fields.get('priority', {}).get('name', None) if fields.get('priority') else None,
        'epic_key': epic_key_linked,
        'parent_key': parent_key,
        'story_points': story_points,
        'labels': labels,
        'components': components
    }
    
    return issue_info


In [ ]:
# IMPROVED get_all_issues with full details
# If get_all_issues is only returning IDs, use this improved version:

def get_all_issues_with_details(project_key, epic_key=None):
    """
    Get all issues with full details for a given project key, optionally filtered by epic.
    This version fetches full details for each issue.
    
    Args:
        project_key (str): The Jira project key (e.g., 'MH')
        epic_key (str, optional): The epic key to filter issues (e.g., 'MH-7'). 
                                  If None, returns all issues in the project.
    
    Returns:
        list: List of dictionaries containing detailed issue information
    """
    import json
    
    # Build JQL query
    if epic_key:
        # Filter by project and epic
        jql_query = f'project = {project_key} AND "Epic Link" = {epic_key}'
    else:
        # Get all issues in project (excluding epics)
        jql_query = f'project = {project_key} AND issuetype != Epic'
    
    # Step 1: Search for issue keys
    url = f"{jira_base_url}/rest/api/3/search/jql"
    headers = {
        "Accept": "application/json"
    }
    auth = (jira_email, jira_api_key)
    query = {
        'jql': jql_query,
        'fields': 'key'  # Only get keys first
    }
    
    all_issue_keys = []
    start_at = 0
    max_results = 100
    
    # Handle pagination to get all issue keys
    while True:
        query['startAt'] = start_at
        query['maxResults'] = max_results
        
        response = requests.request(
            "GET",
            url,
            headers=headers,
            params=query,
            auth=auth
        )
        response.raise_for_status()
        search_data = json.loads(response.text)
        
        issues = search_data.get('issues', [])
        if not issues:
            break
        
        # Collect issue keys
        for issue in issues:
            all_issue_keys.append(issue.get('key'))
        
        # Check if there are more results
        total = search_data.get('total', 0)
        start_at += len(issues)
        if start_at >= total:
            break
    
    # Step 2: Get full details for each issue
    print(f"Found {len(all_issue_keys)} issues. Fetching details...")
    all_issues_details = []
    for i, issue_key in enumerate(all_issue_keys):
        print(f"Fetching {i+1}/{len(all_issue_keys)}: {issue_key}")
        issue_details = get_issue_details(issue_key)
        all_issues_details.append(issue_details)
    
    return all_issues_details


In [ ]:
# Test the function
epics = get_all_epics('MH')
print(f"Found {len(epics)} epic(s)\n")
for epic in epics:
    print(json.dumps(epic, indent=2))
    print("-" * 50)


In [ ]:
project_key = 'MH'
epic_key = 'MH-7'
issue_type = 'Task'
summary = 'Test Story 3'
description = 'This is a test story'
story_points = 1
start_date = '2025-12-01'
due_date = '2025-12-05'


create_jira_issue(project_key, epic_key, summary, description, issue_type,
                  parent_key=None, start_date='2025-12-01', due_date='2025-12-05',
                  story_points=1)


In [ ]:
update_status = update_jira_issue(issue_key='MH-10', summary='Test is updated test issue', status='Done')


In [ ]:
all_issues_ids = get_all_issues(project_key, epic_key)
all_issues_ids

In [ ]:
get_issue_details(all_issues_ids[0]['id'])

In [ ]:
all_issues_details = get_all_issues_with_details(project_key, epic_key)

In [ ]:
all_issues_details[-1]